# RepairScope-X — Notebook 25 v3 | Guided Independent Adjudication

**710 original trials · 159 flagged cases · No GPU · Fully embedded evidence**

This version adds a per-case interactive reviewer panel, fingerprinted record identity, independent Reviewer A/B vote files, optional third-reviewer arbitration, a corrected partial-identification bound, and publication-ready reconciliation outputs.

**Important:** Automated checks are aids, **not human reviews**. Initially **159/159 cases now have human consensus or third-reviewer arbitration (subject to final import validation)**. Do not treat the labels as validated until two independent people review every case and disputes are arbitrated.

### Suggested workflow

1. Run through **Section 3** to reconstruct the frozen records and blank templates.
2. In **Section 4**, enter your reviewer code and choose A or B. Use the panel to read each original prompt, generated response, original wrong response, and grader evidence. Save decisions one at a time. Each reviewer must work independently and use a different identity.
3. Reviewer A and Reviewer B use different saved CSV files; share the two completed files only after independent review. Saving one person's file does **not** complete dual review.
4. Set `REVIEW_A_PATH` and `REVIEW_B_PATH` in **Section 5**, and rerun Sections 5–7. Where A/B disagree, an independent Reviewer C can use the arbitration worksheet and the `ARBITRATION_PATH` import.
5. Export the results ZIP. Until human review and the pending Notebook 21 MDLM repair results are incorporated, **full manuscript freeze remains blocked**.

For best persistence on Colab, **mount Google Drive before running the setup cell**. Otherwise reviewer CSVs are saved to the session filesystem and should be downloaded before the session ends.


### Third-reviewer adjudication imported (October 19, 2026)

This notebook includes the independently submitted Reviewer A and B decisions and Reviewer C's three final judgments for cases 6, 147 and 148. All three C judgments are (task=0, expression=0, strict-final=0). These rows are matched to the **exact cohort-qualified record IDs and SHA-256 record fingerprints**.

The 159 flagged cases can be marked adjudicated when validation passes. **The remaining 551 unflagged trials are not individually human-reviewed**, and Notebook 21's 64-trial MDLM repair stage remains unavailable. Treat publication metrics as partly adjudicated until remaining methodological work is complete.

A/B/C reviewer submissions are embedded and checksum-checked, so this notebook runs without uploading any CSV files. It will reconstruct reviewer files into its output folder. The input evidence remains embedded in the notebook.

In [ ]:
OUTPUT_FOLDER='RepairScopeX_NB25_Final_Arbitrated_20261019'

from pathlib import Path
import base64,io,zipfile,hashlib,json,csv,math,random,statistics,re,ast,collections
from collections import Counter,defaultdict
from fractions import Fraction
from datetime import datetime, timezone
try:
    from google.colab import files as colab_files
except ImportError:
    colab_files=None
OUTPUT_ROOT=Path('/content' if Path('/content').exists() else '/mnt/data')
OUT=OUTPUT_ROOT/OUTPUT_FOLDER
OUT.mkdir(parents=True,exist_ok=True)
print('Results directory:',OUT)

In [ ]:
# Frozen, reviewer-submitted decision forms are embedded for reproducible local/Colab execution.
# Review provenance: reviewer IDs and fingerprints checked against the 710 frozen records.
VOTES_B64='...'
VOTES_SHA256={'Reviewer_A_BLINDED_VOTES_completed.csv': 'df799c38da4dbca19d0c78e8ad1a7049a9cc38fd020aa6e2bfbcebcd280fddaa', 'Reviewer_B_BLINDED_VOTES_completed.csv': 'e9f5e3d371b124d6556b40944520278df900ce104e436e271073ca54fc59b5cf', 'Reviewer_C_3_Cases_COMPLETED.csv': '35856b07b7f07b9d775128c3d7f29c35c46a9858e3f6c489625a58c61f1bdbaa'}
with zipfile.ZipFile(io.BytesIO(base64.b64decode(VOTES_B64))) as _votezip:
    for _name,_digest in VOTES_SHA256.items():
        _raw=_votezip.read(_name)
        assert hashlib.sha256(_raw).hexdigest()==_digest, 'Embedded reviewer votes checksum mismatch: '+_name
        (OUT/_name).write_bytes(_raw)
print('Embedded A/B/C reviewer sheets verified:',len(VOTES_SHA256))


In [ ]:
BUNDLE_SHA256='ce4301a1201354e57b7c247fec4f24870ccc5d2edc854fa726e51d639861905e'
BUNDLE_B64='''...'''

raw_bundle=base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(raw_bundle).hexdigest()==BUNDLE_SHA256
with zipfile.ZipFile(io.BytesIO(raw_bundle)) as z:
    INPUTS={k:z.read(k) for k in z.namelist()}
assert len(INPUTS)==16
PROVENANCE=json.loads(INPUTS['provenance.json'])
for key,record in PROVENANCE.items():
    ext='csv' if record['kind']=='csv' else 'jsonl'
    assert hashlib.sha256(INPUTS[f'{key}.{ext}']).hexdigest()==record['sha256'], key
print('Evidence provenance verified:',len(PROVENANCE),'original evidence files')

### 1. Reconstruct and validate raw evidence

In [ ]:
def load_jsonl(key):
    return [json.loads(line) for line in INPUTS[key+'.jsonl'].decode().splitlines() if line.strip()]
def write_csv(name,rows):
    p=OUT/name
    cols=list(dict.fromkeys(k for r in rows for k in r)) if rows else ['empty']
    with p.open('w',encoding='utf-8',newline='') as h:
        w=csv.DictWriter(h,fieldnames=cols);w.writeheader()
        for row in rows:
            w.writerow({k:json.dumps(v,ensure_ascii=False) if isinstance(v,(dict,list)) else v for k,v in row.items()})
    return p
def write_json(name,obj):
    p=OUT/name;p.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False,default=str),encoding='utf-8');return p
COHORTS={
 'dream12':('Dream 7B','Dream exploratory','4-bit NF4','mask reopening'),
 'dream13':('Dream 7B','Dream independent','4-bit NF4','mask reopening'),
 'alodlm15':('ALoDLM 1.7B','ALoDLM entropy','fp32','prefix-conditioned suffix'),
 'alodlm16_left1':('ALoDLM 1.7B','ALoDLM left1','fp32','prefix-conditioned suffix'),
 'mdlm18':('MDLM 0.5B','MDLM 64-token','fp16','native infill'),
 'bd3lm19':('BD3LM 0.5B','BD3LM 64-token','fp16','prefix-conditioned suffix'),
 'bd3lm23':('BD3LM 0.5B','BD3LM 32-step/block','fp16','prefix-conditioned suffix'),
}
REPAIR_KEYS=list(COHORTS)
raw={k:load_jsonl(k) for k in REPAIR_KEYS}
expected={'dream12':240,'dream13':150,'alodlm15':64,'alodlm16_left1':64,'mdlm18':64,'bd3lm19':64,'bd3lm23':64}
assert {k:len(v) for k,v in raw.items()}==expected
assert sum(map(len,raw.values()))==710
natural={k:load_jsonl(k) for k in ['dream11natural','alodlm14natural','mdlm18natural','bd3lm19natural','mdlm21natural','bd3lm23natural','mdlm20length']}
prompts=defaultdict(list)
for src,records in natural.items():
    for x in records:
        if x.get('problem_id') and x.get('prompt'):
            prompts[x['problem_id']].append((src,x['prompt'],x.get('target')))
ledger=[]
for key,records in raw.items():
    for x in records:
        v=x.get('verification') or {}
        target=v.get('target',x.get('target'))
        if target is None:
            for _,_,t in prompts.get(x['problem_id'],[]):
                if t is not None:target=t;break
        id_=x.get('trial_id') or x.get('id')
        assert id_ and x.get('status')=='ok'
        native_nfe=x.get('actual_forward_calls',x.get('nfe'))
        orig=x.get('original_output','')
        repaired=x.get('repaired_output',x.get('output',''))
        role=x.get('confirmatory_role','') if key=='dream13' else ('exploratory' if key=='dream12' else 'model-native')
        ledger.append({'cohort_key':key,'cohort':COHORTS[key][1],'model':COHORTS[key][0],
           'precision':COHORTS[key][2],'intervention':COHORTS[key][3],
           'trial_id':id_,'problem_id':x['problem_id'],'dataset':x['dataset'],'role':role,
           'action':x['action'],'reported_task_success':int(bool(x.get('task_success'))),
           'reported_numeric_success':int(bool(x.get('numeric_success'))),
           'reported_strict_success':int(bool(x.get('strict_success'))),
           'original_output':orig,'repaired_output':repaired,'target':target,
           'token_preservation':x.get('token_position_preservation',x.get('preservation')),
           'forward_calls':native_nfe,'loop_token_passes':x.get('loop_token_passes'),
           'seconds':x.get('wall_seconds'), 'raw_record':x})
assert len(ledger)==710
for key in REPAIR_KEYS:
    ids=[x['trial_id'] for x in ledger if x['cohort_key']==key]
    assert len(ids)==len(set(ids)),f'Duplicate trial ID in {key}'
print('710 verified repair trials across seven non-overlapping cohort ledgers')

### 2. Four well-defined evaluation views and unit tests

In [ ]:
FINAL_ANY=re.compile(r'(?im)^[ \t]*FINAL_ANSWER\s*:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$')
NUMBER=re.compile(r'(?<![A-Za-z0-9_])[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?(?![A-Za-z0-9_])')
PROMPT_INPUT=re.compile(r'Use the numbers\s+(.*?)\s+exactly once',re.I|re.S)
# Conservative: arithmetic sequences with >= 2 operands; supports simple parentheses and the operators +,-,*,/.
EXPR_CAND=re.compile(r'(?<![\w.])(?:\(?\s*[-+]?\d+(?:\.\d+)?\s*\)?\s*[-+*/]\s*)+\(?\s*[-+]?\d+(?:\.\d+)?\s*\)?(?![\w.])')
def floats_equal(a,b):
    try:return math.isfinite(float(a)) and math.isfinite(float(b)) and math.isclose(float(a),float(b),rel_tol=0,abs_tol=1e-8)
    except (TypeError,ValueError,OverflowError):return False

def safe_arithmetic(expression):
    # AST whitelist, rational math, bounded complexity; NEVER eval() model text.
    try:tree=ast.parse(expression,mode='eval')
    except (SyntaxError,ValueError,TypeError):return None
    if len(list(ast.walk(tree)))>64:return None
    nums=[]
    def walk(n):
        if isinstance(n,ast.Expression):return walk(n.body)
        if isinstance(n,ast.Constant) and type(n.value) in (float,int):
            if not math.isfinite(float(n.value)) or abs(float(n.value))>1e6:raise ValueError()
            nums.append(Fraction(str(n.value)));return nums[-1]
        if isinstance(n,ast.UnaryOp) and isinstance(n.op,(ast.UAdd,ast.USub)):
            v=walk(n.operand);return v if isinstance(n.op,ast.UAdd) else -v
        if isinstance(n,ast.BinOp) and isinstance(n.op,(ast.Add,ast.Sub,ast.Mult,ast.Div)):
            a,b=walk(n.left),walk(n.right)
            c=a+b if isinstance(n.op,ast.Add) else a-b if isinstance(n.op,ast.Sub) else a*b if isinstance(n.op,ast.Mult) else a/b
            if abs(c.numerator)>10**15 or c.denominator>10**15:raise ValueError()
            return c
        raise ValueError()
    try:return walk(tree),nums
    except (ValueError,ZeroDivisionError,TypeError,OverflowError):return None

def source_numbers(rec):
    ver=rec['raw_record'].get('verification') or {}
    struct=ver.get('countdown_structured') or {}
    src=struct.get('source_numbers')
    if src:
        return [Fraction(str(a)) for a in src],'original_verifier'
    for source,prompt,_ in prompts.get(rec['problem_id'],[]):
        m=PROMPT_INPUT.search(prompt)
        if m:
            nums=[Fraction(a) for a in NUMBER.findall(m.group(1))]
            if nums:return nums,'natural_prompt:'+source
    return None,'unresolved'

def find_expression(text,nums,target):
    if nums is None or target is None:return '',0
    normalized=text.replace('×','*').replace('÷','/').replace('−','-')
    for match in EXPR_CAND.finditer(normalized):
        cand=match.group(0).strip()
        parsed=safe_arithmetic(cand)
        if parsed is None:continue
        val,found=parsed
        if len(found)==len(nums) and Counter(found)==Counter(nums) and floats_equal(val,target):
            return cand,1
    return '',0

def analyze_trial(row):
    out=row['repaired_output'] or ''
    matches=FINAL_ANY.findall(out)
    explicit_correct=int(bool(matches) and floats_equal(matches[-1],row['target']))
    # Strict final line must not be markdown/backtick-wrapped.
    last_line=next((line.strip() for line in reversed(out.splitlines()) if line.strip()),'')
    ending=FINAL_ANY.fullmatch(last_line)
    strict_correct=int(bool(ending) and floats_equal(ending.group(1),row['target']))
    nums,src=source_numbers(row) if row['dataset']=='countdown_fresh' else (None,'not_applicable')
    witness,witness_yes=find_expression(out,nums,row['target']) if row['dataset']=='countdown_fresh' else ('',0)
    # These are independent diagnostics, not the original benchmark's decision rule.
    auto_task=int(bool(witness_yes)) if row['dataset']=='countdown_fresh' else explicit_correct
    format_and_evidence=int(bool(strict_correct and (witness_yes if row['dataset']=='countdown_fresh' else True)))
    return {'trial_id':row['trial_id'],'cohort':row['cohort'],'cohort_key':row['cohort_key'],
      'role':row['role'],'problem_id':row['problem_id'],'dataset':row['dataset'],
      'action':row['action'],'target':row['target'],
      'reported_task_success':row['reported_task_success'],
      'reported_numeric_success':row['reported_numeric_success'],
      'explicit_final_any':int(bool(matches)),'explicit_final_numeric_correct':explicit_correct,
      'strict_final_line_correct':strict_correct,
      'expression_witness':witness,'expression_witness_found':witness_yes,
      'source_numbers':','.join(map(str,nums)) if nums is not None else '',
      'source_numbers_origin':src,
      'automated_evidence_based_success':auto_task,
      'automated_format_and_evidence_success':format_and_evidence,
      'reported_vs_automated_disagreement':int(row['reported_task_success']!=auto_task),
      'original_output':row['original_output'],'repaired_output':out}

assert safe_arithmetic('__import__("os")') is None
assert safe_arithmetic('2 + 24 + 28')[0]==54
assert safe_arithmetic('29+9+8')[0]!=12
assert find_expression('2 + 24 + 28 = 54',[Fraction(2),Fraction(24),Fraction(28)],54)[1]==1
assert find_expression('229 - 99 + 8 = 12',[Fraction(29),Fraction(9),Fraction(8)],12)[1]==0
assert find_expression('8 / (2 - 2) = 4',[Fraction(8),Fraction(2),Fraction(2)],4)[1]==0
print('Safe arithmetic and mismatch unit checks: passed')

### 3. Reconcile the manual-review queue (no automatic adjudication)

In [ ]:
audit=[analyze_trial(rec) for rec in ledger]
assert len(audit)==710
for row in audit:
    row['review_id']=row['cohort_key']+'::'+row['trial_id']
assert len({r['review_id'] for r in audit})==710, 'Composite review IDs must uniquely identify all trials'
write_csv('audit_710_multirubric.csv',audit)
original_review=list(csv.DictReader(io.StringIO(INPUTS['manual140.csv'].decode('utf-8'))))
assert len(original_review)==140 and len({(r['cohort'],r['trial_id']) for r in original_review})==140
cohort_name_to_key={metadata[1]:key for key,metadata in COHORTS.items()}
byid={r['review_id']:r for r in audit}
review_reasons=defaultdict(set)
for historical in original_review:
    ckey=cohort_name_to_key.get(historical['cohort'])
    assert ckey, 'Unknown historical cohort: '+historical['cohort']
    rid=ckey+'::'+historical['trial_id']
    assert rid in byid, 'Historical review record not found: '+rid
    review_reasons[rid].add('historical_review_queue')
for r in audit:
    rid=r['review_id']
    if r['reported_vs_automated_disagreement']:
        review_reasons[rid].add('reported_vs_automated_evidence')
    if r['reported_task_success'] and not r['strict_final_line_correct']:
        review_reasons[rid].add('reported_success_missing_strict_final')
    if r['reported_numeric_success']!=r['explicit_final_numeric_correct']:
        review_reasons[rid].add('numeric_vs_explicit_final_disagreement')
    if r['dataset']=='countdown_fresh' and r['source_numbers_origin']=='unresolved':
        review_reasons[rid].add('countdown_source_missing')
review_ids=sorted(review_reasons)
review_sheet=[]
for rid in review_ids:
    r=byid[rid]
    review_sheet.append({**r,'review_reasons':';'.join(sorted(review_reasons[rid])),
       'reviewer_id':'','human_task_success':'','human_expression_valid':'',
       'human_strict_final_valid':'','evidence_note':''})
assert len({r['review_id'] for r in review_sheet})==len(review_sheet)
write_csv('review_template_A_UNFILLED.csv',review_sheet)
write_csv('review_template_B_UNFILLED.csv',review_sheet)
write_csv('review_reason_table.csv',[{'reason':reason,'n_trials':sum(reason in values for values in review_reasons.values())} for reason in sorted(set.union(*(v for v in review_reasons.values())))])
print('Historical manual queue:',len(original_review))
print('Deduplicated cohort-qualified review queue:',len(review_sheet))
print('Trial_id collisions in complete audit:',len(audit)-len({r['trial_id'] for r in audit}))
print('Reported task successes:',sum(x['reported_task_success'] for x in audit))
print('Conservative automated evidence successes:',sum(x['automated_evidence_based_success'] for x in audit))
print('Automated full-format-and-evidence successes:',sum(x['automated_format_and_evidence_success'] for x in audit))

# Distinguish the original benchmark source from the current repaired response.
PROMPT_SOURCE_MAP={
    'dream12':'dream11natural','dream13':'dream11natural',
    'alodlm15':'alodlm14natural','alodlm16_left1':'alodlm14natural',
    'mdlm18':'mdlm18natural','bd3lm19':'bd3lm19natural',
    'bd3lm23':'bd3lm23natural',
}
source_prompts={}
for ckey,source in PROMPT_SOURCE_MAP.items():
    for trial in natural[source]:
        pid=trial.get('problem_id')
        if pid and trial.get('prompt'):
            source_prompts[(ckey,pid)]=trial['prompt']
for r in audit:
    r['source_prompt']=source_prompts.get((r['cohort_key'],r['problem_id']),'')
    canonical = json.dumps({k:r[k] for k in (
        'cohort_key','trial_id','problem_id','dataset','action','target',
        'original_output','repaired_output')},sort_keys=True,ensure_ascii=False,default=str)
    r['record_fingerprint']=hashlib.sha256(canonical.encode('utf-8')).hexdigest()
assert len({r['record_fingerprint'] for r in audit})==710
# Rebuild keys/worksheets with fingerprints and actual source prompts.
byid={r['review_id']:r for r in audit}
review_sheet=[]
for rid in review_ids:
    r=byid[rid]
    review_sheet.append({**r,'review_reasons':';'.join(sorted(review_reasons[rid])),
      'reviewer_id':'','human_task_success':'','human_expression_valid':'',
      'human_strict_final_valid':'','evidence_note':''})
write_csv('audit_710_multirubric.csv',audit)
write_csv('review_template_A_UNFILLED.csv',review_sheet)
write_csv('review_template_B_UNFILLED.csv',review_sheet)
assert all(byid[r['review_id']]['record_fingerprint']==r['record_fingerprint'] for r in review_sheet)
print('Fingerprint-bound reviewer templates:',len(review_sheet),'| source prompts present:',sum(bool(r['source_prompt']) for r in review_sheet))


### 4. Guided review panel (one real reviewer per CSV)

This panel displays the **original model prompt**, native failed answer, repaired response, expected target, and all automated diagnostics. It does **not** create votes by itself. A vote is saved only when the reviewer clicks **Save decision** and supplies all three labels plus a short evidence note.

- For **Countdown**, task validity requires an arithmetic expression using the stated numbers exactly once and reaching the target; do not count the target appearing incidentally in an invalid expression.
- For **GSM8K/SVAMP**, assess the requested numeric answer using the original problem and its evidence. Differentiate a plausible number from a completed answer.
- Mark **strict-final-format** separately: a last nonblank line exactly `FINAL_ANSWER: <number>` with the correct number.
- Where the available evidence is insufficient, select **uncertain**; do not guess. This triggers arbitration after dual review.
- **Reviewer A and Reviewer B must use different reviewer codes**, and ideally separate Colab sessions. The save file includes a fingerprint to prevent accidental cross-model assignment.

Automated labels are hidden by default to reduce reviewer anchoring, and can be revealed afterward. The notebook does not contact a model or use automated labels as reviewer votes. If widgets are unavailable, fill the generated CSV template offline instead.

In [ ]:
# Change before working. Separate reviewers MUST use distinct codes and distinct files.
REVIEWER_ROLE='A'      # Choose 'A' or 'B'. For arbitrator C, use the import workflow below.
REVIEWER_CODE=''        # e.g., 'reviewer_01' (do not use the same code as your co-reviewer)

from datetime import datetime, timezone
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    HAS_WIDGETS=True
    if 'google.colab' in __import__('sys').modules:
        try:
            from google.colab import output as _colab_output
            _colab_output.enable_custom_widget_manager()
        except Exception:
            pass
except ImportError:
    HAS_WIDGETS=False

DRIVE_REVIEWS=Path('/content/drive/MyDrive/RepairScopeX/RSX25_Independent_Reviews')
REVIEW_SAVE_DIR = DRIVE_REVIEWS if Path('/content/drive/MyDrive').is_dir() else OUT
REVIEW_SAVE_DIR.mkdir(parents=True, exist_ok=True)

REVIEW_COLUMNS=['review_id','record_fingerprint','cohort_key','trial_id','problem_id','reviewer_id',
    'human_task_success','human_expression_valid','human_strict_final_valid','evidence_note','saved_at_utc']

def _review_path(role,who):
    if role not in ('A','B'):raise ValueError('Use role A/B for independent review; arbitration is imported separately')
    if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]{2,63}',str(who)):
        raise ValueError('Set REVIEWER_CODE to at least 3 safe characters (letters/digits/underscore/dot/dash).')
    return REVIEW_SAVE_DIR / ('RSX25_REVIEW_'+role+'_'+who+'.csv')

def _read_saved_reviews(path):
    if not path.exists():return {}
    with path.open('r',encoding='utf-8-sig',newline='') as f:
        rows=list(csv.DictReader(f))
    found={}
    for item in rows:
        rid=item.get('review_id','')
        if rid not in byid:raise ValueError('Saved review has unknown review_id: '+rid)
        ref=byid[rid]
        for k in ('record_fingerprint','cohort_key','trial_id','problem_id'):
            if item.get(k,'')!=str(ref[k]):raise ValueError(f'Saved review mismatched {k} for '+rid)
        if rid in found:raise ValueError('Duplicate saved review_id: '+rid)
        found[rid]=item
    return found

def save_review_decision(role,who,rid,task,expression,strict,note):
    path=_review_path(role,who)
    if rid not in review_reasons:raise ValueError('Unknown case for review')
    if any(v not in ('0','1','uncertain') for v in (task,expression,strict)):
        raise ValueError('Each label must be 0, 1, or uncertain')
    if not str(note).strip() or len(note.strip())<8:
        raise ValueError('Provide at least 8 characters of supporting evidence')
    old=_read_saved_reviews(path)
    if any(x.get('reviewer_id')!=who for x in old.values()):
        raise ValueError('Saved file belongs to another reviewer; use a separate reviewer code')
    old[rid]={
        'review_id':rid,'record_fingerprint':byid[rid]['record_fingerprint'],
        'cohort_key':byid[rid]['cohort_key'],'trial_id':byid[rid]['trial_id'],
        'problem_id':byid[rid]['problem_id'],'reviewer_id':who,
        'human_task_success':task,'human_expression_valid':expression,
        'human_strict_final_valid':strict,'evidence_note':note.strip(),
        'saved_at_utc':datetime.now(timezone.utc).isoformat()
    }
    temp=path.with_suffix('.tmp')
    with temp.open('w',encoding='utf-8',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=REVIEW_COLUMNS)
        writer.writeheader()
        writer.writerows(old[r] for r in sorted(old))
    temp.replace(path)
    return path,len(old)

if not HAS_WIDGETS:
    print('ipywidgets unavailable. Use blank CSV templates instead; review files must contain the v3 record_fingerprint column.')
elif not REVIEWER_CODE:
    print('To start the panel, set REVIEWER_CODE to your reviewer pseudonym, then rerun this cell.')
else:
    review_path=_review_path(REVIEWER_ROLE,REVIEWER_CODE)
    saved_now=_read_saved_reviews(review_path)
    current_index=[0]
    unreviewed_only=widgets.Checkbox(value=True,description='Unreviewed only')
    progress=widgets.HTML()
    show_diagnostics=widgets.Checkbox(value=False,description='Reveal automated labels AFTER independent judgment')
    preview=widgets.Output(layout=widgets.Layout(border='1px solid #bbb',max_height='550px',overflow='auto',padding='8px'))
    message=widgets.Output()
    task_value=widgets.Dropdown(options=[('Select…',''),('Valid','1'),('Invalid','0'),('Uncertain','uncertain')],description='Task')
    expr_value=widgets.Dropdown(options=[('Select…',''),('Valid','1'),('Invalid','0'),('Uncertain','uncertain')],description='Expression')
    strict_value=widgets.Dropdown(options=[('Select…',''),('Valid','1'),('Invalid','0'),('Uncertain','uncertain')],description='Final format')
    evidence=widgets.Textarea(placeholder='Specific output evidence and why this ruling follows the task rubric',description='Evidence',layout=widgets.Layout(width='95%',height='100px'))
    btn_back=widgets.Button(description='Previous')
    btn_next=widgets.Button(description='Next')
    btn_save=widgets.Button(description='Save decision',button_style='success')
    btn_unreviewed=widgets.Button(description='Next unreviewed')
    case_jump=widgets.BoundedIntText(value=1,min=1,max=len(review_ids),description='Case #',layout=widgets.Layout(width='160px'))
    btn_jump=widgets.Button(description='Go')

    def _refresh():
        idx=current_index[0]
        rid=review_ids[idx]
        case=byid[rid]
        progress.value=f'<b>Reviewer {REVIEWER_ROLE}: {REVIEWER_CODE}</b> | Case {idx+1}/{len(review_ids)} | Saved: {len(saved_now)} | <code>{rid}</code>'
        case_jump.value=idx+1
        old=saved_now.get(rid,{})
        for widget_,key in [(task_value,'human_task_success'),(expr_value,'human_expression_valid'),(strict_value,'human_strict_final_valid')]:
            widget_.value=old.get(key,'')
        evidence.value=old.get('evidence_note','')
        with preview:
            clear_output(wait=True)
            print('Model:',case['cohort'],'| Dataset:',case['dataset'],'| Action:',case['action'])
            print('Problem ID:',case['problem_id'],'| Target:',case['target'])
            print('Review flags:',', '.join(sorted(review_reasons[rid])))
            print('\nORIGINAL PROMPT (as recorded):\n',case['source_prompt'] or '[Not included in frozen source bundle]')
            print('\nORIGINAL FAILED OUTPUT:\n',case['original_output'] or '[Not recorded]')
            print('\nREPAIRED OUTPUT:\n',case['repaired_output'] or '[Empty]')
            if show_diagnostics.value:
                print('\nAUTOMATED DIAGNOSTICS — suggestions, NOT ground truth')
                for name in ['reported_task_success','reported_numeric_success','expression_witness','source_numbers','source_numbers_origin','explicit_final_numeric_correct','strict_final_line_correct','automated_evidence_based_success','automated_format_and_evidence_success']:
                    if name in case: print(name+':',case.get(name))

    def _back(_):
        current_index[0]=max(0,current_index[0]-1);_refresh()
    def _next(_):
        current_index[0]=min(len(review_ids)-1,current_index[0]+1);_refresh()
    def _go(_):
        current_index[0]=case_jump.value-1;_refresh()
    def _next_unreviewed(_):
        j=next((k for k in range(current_index[0]+1,len(review_ids)) if review_ids[k] not in saved_now),None)
        if j is None:j=next((k for k in range(len(review_ids)) if review_ids[k] not in saved_now),None)
        with message:
            clear_output(wait=True)
            if j is None:print('All assigned cases have saved votes.')
        if j is not None:current_index[0]=j;_refresh()
    def _save(_):
        rid=review_ids[current_index[0]]
        try:
            path,n=save_review_decision(REVIEWER_ROLE,REVIEWER_CODE,rid,
                task_value.value,expr_value.value,strict_value.value,evidence.value)
            saved_now.update(_read_saved_reviews(path))
            with message:
                clear_output(wait=True)
                print('Saved',n,'/',len(review_ids),'to',path)
            _refresh()
        except Exception as exc:
            with message:
                clear_output(wait=True)
                print('NOT saved:',type(exc).__name__,str(exc))
    for b,fn in [(btn_back,_back),(btn_next,_next),(btn_save,_save),(btn_unreviewed,_next_unreviewed),(btn_jump,_go)]:
        b.on_click(fn)
    show_diagnostics.observe(lambda ch: _refresh(),names='value')
    display(widgets.VBox([progress,widgets.HBox([btn_back,btn_next,btn_unreviewed,case_jump,btn_jump]),show_diagnostics,
        preview,widgets.HBox([task_value,expr_value,strict_value]),evidence,btn_save,message]))
    _refresh()
    print('Reviewer file:',review_path)
    print('Keep this file for import. If using Colab and Drive is not mounted, download the CSV before session ends.')

In [ ]:
# Self-test the saving/import identity protections using an isolated temp folder.
import tempfile
from pathlib import Path
_test_parent=REVIEW_SAVE_DIR
with tempfile.TemporaryDirectory(prefix='rsx25_review_safety_') as td:
    REVIEW_SAVE_DIR=Path(td)
    rid=review_ids[0]
    testfile,n=save_review_decision('A','test_safety',rid,'0','uncertain','0',
        'Synthetic test vote: do not use as a human judgment')
    assert n==1
    check=_read_saved_reviews(testfile)
    assert list(check)==[rid]
    # Tamper with fingerprint and verify the error is caught.
    with testfile.open(encoding='utf-8',newline='') as f:rows=list(csv.DictReader(f))
    rows[0]['record_fingerprint']='corrupted'
    with testfile.open('w',encoding='utf-8',newline='') as f:
        w=csv.DictWriter(f,fieldnames=REVIEW_COLUMNS);w.writeheader();w.writerows(rows)
    try:_read_saved_reviews(testfile)
    except ValueError:pass
    else:raise AssertionError('Corrupted review fingerprint was accepted')
REVIEW_SAVE_DIR=_test_parent
print('Reviewer saving, resume, and fingerprint rejection self-tests: PASSED')

#### Alternative: keyboard-based review (no widgets needed)

After setting `REVIEWER_ROLE` and `REVIEWER_CODE` in Section 4, use the helpers below if widgets cannot render. For example, `show_review_case(1)` displays the first response; `submit_review_case(1,'0','0','0','The expression omits one of the source numbers')` records an independent human ruling. Reviewer files persist in Google Drive if it was mounted before setup.

In [ ]:
def show_review_case(case_number,show_auto=False):
    idx=int(case_number)-1
    if not 0<=idx<len(review_ids):raise IndexError(f'Choose case 1–{len(review_ids)}')
    r=byid[review_ids[idx]]
    print(f'CASE {idx+1}/{len(review_ids)} | {r["review_id"]} | {r["cohort"]} | {r["dataset"]}')
    print('Problem:',r['problem_id'],'| Target:',r['target'])
    print('\nORIGINAL PROMPT\n',r.get('source_prompt','') or '[not available]')
    print('\nORIGINAL FAILURE\n',r['original_output'] or '[not recorded]')
    print('\nREPAIRED OUTPUT\n',r['repaired_output'] or '[empty]')
    if show_auto:
        for k in ('reported_task_success','expression_witness','explicit_final_numeric_correct','automated_evidence_based_success'):
            print(k,':',r.get(k))
    return r['review_id']
def submit_review_case(case_number,task,expression,strict,evidence_note):
    rid=review_ids[int(case_number)-1]
    saved,n=save_review_decision(REVIEWER_ROLE,REVIEWER_CODE,rid,
        str(task),str(expression),str(strict),evidence_note)
    print('Saved',n,'review(s) to',saved)
    return saved
def remaining_review_cases():
    saved=_read_saved_reviews(_review_path(REVIEWER_ROLE,REVIEWER_CODE))
    pending=[i+1 for i,rid in enumerate(review_ids) if rid not in saved]
    print('Completed:',len(saved),'of',len(review_ids),'| Next pending:',pending[:10])
    return pending
print('Keyboard fallback ready: show_review_case(1); submit_review_case(...); remaining_review_cases()')

### 5. Merge independent votes and resolve disputes

Set the two reviewer file paths in the next cell. The notebook does **not** infer reviewer votes from automated graders. It blocks votes that refer to different model cohorts, fingerprints, or problem IDs. Two separate reviewer codes are required; unresolved cases stay unresolved. A third reviewer can resolve disputed cases after completing the exported `arbitration_required.csv` worksheet.


In [ ]:
# Set these to the TWO separately completed reviewer CSVs before finalizing.
# They may point to files created by the panel above or externally completed v3 templates.
REVIEW_A_PATH=str(OUT/'Reviewer_A_BLINDED_VOTES_completed.csv')
REVIEW_B_PATH=str(OUT/'Reviewer_B_BLINDED_VOTES_completed.csv')
ARBITRATION_PATH=str(OUT/'Reviewer_C_3_Cases_COMPLETED.csv')    # Optional independently completed third-reviewer sheet for disputed cases.

REVIEW_KEYS=('human_task_success','human_expression_valid','human_strict_final_valid')
def _load_review_votes(path,tag):
    if not path:return {}
    filename=Path(path)
    if not filename.is_file():raise FileNotFoundError(filename)
    with filename.open(encoding='utf-8-sig',newline='') as f:rows=list(csv.DictReader(f))
    votes={}; reviewers=set()
    for row in rows:
        rid=row.get('review_id','').strip()
        labels=tuple(row.get(k,'').strip().lower() for k in REVIEW_KEYS)
        if not rid and all(x=='' for x in labels):continue
        if rid not in review_reasons:raise ValueError(f'{tag}: unknown review_id: '+rid)
        if rid in votes:raise ValueError(f'{tag}: duplicate review_id: '+rid)
        ref=byid[rid]
        for field in ('cohort_key','trial_id','problem_id','record_fingerprint'):
            if row.get(field,'').strip()!=str(ref[field]):
                raise ValueError(f'{tag}: {field} mismatch for {rid} — use v3 templates')
        if any(x not in ('0','1','uncertain') for x in labels):
            raise ValueError(f'{tag}: all 3 labels required (0,1,uncertain) for {rid}')
        who=row.get('reviewer_id','').strip()
        note=row.get('evidence_note','').strip()
        if len(who)<3 or len(note)<8:raise ValueError(f'{tag}: reviewer_id and meaningful evidence mandatory for {rid}')
        reviewers.add(who)
        votes[rid]={'reviewer_id':who,'answers':labels,'evidence':note}
    if len(reviewers)>1:raise ValueError(f'{tag}: one reviewer identity per file required')
    return votes
vote_a=_load_review_votes(REVIEW_A_PATH,'reviewer_A')
vote_b=_load_review_votes(REVIEW_B_PATH,'reviewer_B')
vote_c=_load_review_votes(ARBITRATION_PATH,'arbitrator_C')

# Explicitly distinguish two-reviewer agreements from independently arbitrated cases.
status=[];resolved={};needs_arbitration=[]
for rid in review_ids:
    a,b,c=vote_a.get(rid),vote_b.get(rid),vote_c.get(rid)
    if a is None or b is None:
        state='awaiting_two_reviews'
    elif a['reviewer_id']==b['reviewer_id']:
        state='not_independent'
    elif a['answers']!=b['answers'] or 'uncertain' in a['answers']+b['answers']:
        state='needs_arbitration'
    else:
        state='consensus';resolved[rid]=a['answers'][0]
    if state=='needs_arbitration' and c:
        if c['reviewer_id'] in {a['reviewer_id'],b['reviewer_id']}:
            state='arbitration_not_independent'
        elif 'uncertain' in c['answers']:
            state='arbitration_uncertain'
        else:
            state='arbitrated';resolved[rid]=c['answers'][0]
    if state in ('needs_arbitration','not_independent','arbitration_not_independent','arbitration_uncertain'):
        needs_arbitration.append({'review_id':rid,'cohort_key':byid[rid]['cohort_key'],
            'trial_id':byid[rid]['trial_id'],'problem_id':byid[rid]['problem_id'],'record_fingerprint':byid[rid]['record_fingerprint'],
            'dataset':byid[rid]['dataset'],'target':byid[rid]['target'],
            'source_prompt':byid[rid].get('source_prompt',''),
            'original_output':byid[rid].get('original_output',''),
            'repaired_output':byid[rid].get('repaired_output',''),
            'reviewer_id':'','human_task_success':'','human_expression_valid':'','human_strict_final_valid':'',
            'evidence_note':'','dispute_reason':state})
    status.append({'review_id':rid,'cohort_key':byid[rid]['cohort_key'],'trial_id':byid[rid]['trial_id'],
        'state':state,'report_task_success':byid[rid]['reported_task_success'],
        'reviewer_A':a['reviewer_id'] if a else '',
        'reviewer_B':b['reviewer_id'] if b else '',
        'arbitrator_C':c['reviewer_id'] if c else '',
        'resolved_human_task_success':resolved.get(rid,''),
        'votes_A':json.dumps(a['answers']) if a else '',
        'votes_B':json.dumps(b['answers']) if b else '',
        'vote_C':json.dumps(c['answers']) if c else ''})
write_csv('adjudication_status.csv',status)
write_csv('arbitration_required.csv',needs_arbitration)
HUMAN_REVIEW_COMPLETE=bool(len(resolved)==len(review_ids))
write_json('adjudication_gate.json',{
    'total_flagged':len(review_ids),'review_A_votes':len(vote_a),'review_B_votes':len(vote_b),
    'arbitration_votes':len(vote_c),'resolved_cases':len(resolved),
    'unresolved_cases':len(review_ids)-len(resolved),
    'needs_arbitration_cases':len(needs_arbitration),
    'human_review_complete':HUMAN_REVIEW_COMPLETE,
    'notebook21_mdlm_repair_64_trials':'pending',
    'full_manuscript_freeze_ready':False,
    'note':'Full paper freeze additionally requires the pending model-native MDLM repair cohort and final stat refresh.'
})
print('Resolution:',len(resolved),'/',len(review_ids),'| disputed:',len(needs_arbitration))

### 6. Original metrics, review-uncertainty bounds, and reconciled reporting

The original task-success labels remain unchanged. We publish a separate results column only for decisions actually resolved by two independent reviewers (or by third-reviewer arbitration). Where cases are unresolved, the interval shown is a logical identification bound, **not a confidence interval**.


In [ ]:
# Produce two separate results families: reproducible original labels, and evaluator-sensitivity metrics.
primary=[]
for key in REPAIR_KEYS:
    subset=[x for x in audit if x['cohort_key']==key and (key!='dream13' or x['role']=='primary_unseen_problem')]
    for action in sorted({x['action'] for x in subset}):
        ss=[x for x in subset if x['action']==action]
        problems=defaultdict(list)
        for x in ss:problems[x['problem_id']].append(x)
        primary.append({'cohort':ss[0]['cohort'],'cohort_key':key,'role':'primary_unseen_problem' if key=='dream13' else 'cohort as observed',
          'action':action,'trials':len(ss),'independent_problems':len(problems),
          'reported_task_successes':sum(x['reported_task_success'] for x in ss),
          'automated_evidence_witness_successes':sum(x['automated_evidence_based_success'] for x in ss),
          'explicit_final_correct':sum(x['explicit_final_numeric_correct'] for x in ss),
          'strict_format_and_evidence_successes':sum(x['automated_format_and_evidence_success'] for x in ss),
          'trials_flagged_for_review':sum(x['review_id'] in review_reasons for x in ss),
          'problems_any_original_success':sum(any(t['reported_task_success'] for t in v) for v in problems.values())})
write_csv('publication_table_sensitivity_by_cohort_action.csv',primary)
# Conservative partial identification: for each unresolved case, unknown adjudicated task success could be 0 or 1.
bounds=[]
for row in primary:
    subset=[x for x in audit if x['cohort_key']==row['cohort_key'] and x['action']==row['action'] and (row['cohort_key']!='dream13' or x['role']=='primary_unseen_problem')]
    known=sum(int(resolved.get(x['review_id'],x['reported_task_success'])) for x in subset if x['review_id'] not in review_reasons or x['review_id'] in resolved)
    missing=sum(x['review_id'] in review_reasons and x['review_id'] not in resolved for x in subset)
    n=len(subset)
    bounds.append({'cohort':row['cohort'],'action':row['action'],'trials':n,'known_successes':known,
       'unresolved_trials':missing,'minimum_possible_task_success_rate':known/n,'maximum_possible_task_success_rate':(known+missing)/n})
write_csv('partial_identification_unresolved_labels.csv',bounds)
write_json('NB25_GATE.json',{'source_SHA256':BUNDLE_SHA256,'n_trials':len(audit),'cohorts':len(REPAIR_KEYS),
  'original_manual_queue':len(original_review),'deduplicated_review_queue':len(review_ids),
  'dual_reviewed':len(resolved),'human_adjudication_complete':HUMAN_REVIEW_COMPLETE,
  'full_manuscript_freeze_ready':False,
  'primary_original_metrics_preserved':True,'optional_mdlm21_repair':'pending'})
readme=f"""# RepairScope-X Notebook 25 — Evaluation reconciliation\n\nSource SHA256: `{BUNDLE_SHA256}`. **v3 reviewer identifiers use cohort-qualified review_id AND an exact record_fingerprint; do not reuse v1/v2 reviewer templates.**\n\n- The original 710 reported task-success labels are **not modified**.\n- Original manual review queue: 140. Expanded deduplicated review queue: {len(review_ids)}.\n- Fully dual-reviewed and consistent: {len(resolved)}. Unresolved: {len(review_ids)-len(resolved)}.\n- `automated_evidence_based_success` is not strict final-answer compliance. For Countdown it means a conservatively parsed expression uses each input once and equals the target; for other tasks it means a correct explicitly marked final number.\n- `automated_format_and_evidence_success` additionally requires a correct unwrapped `FINAL_ANSWER: number` final line. This is a stricter **new sensitivity metric**, not an undocumented alteration of the original grader.\n- Conservative expression parsing may miss valid parentheses, Unicode math notation or novel answer formats; manual review remains necessary.\n- Two reviewers should independently fill `review_template_A_UNFILLED.csv` and `review_template_B_UNFILLED.csv`, entering 0/1/uncertain for all three fields, their distinct reviewer ID and evidence note. Paths can be set in the adjudication cell and the rest rerun.\n- Primary reported outcomes must not be relabelled as validated until all flagged cases are reconciled.\n- The MDLM Notebook 21 controlled-denoising repair cohort is not available; it is not treated as negative data.\n"""
(OUT/'README_VERIFICATION.md').write_text(readme)
print('NB25 gate:',json.loads((OUT/'NB25_GATE.json').read_text()))

# A resolved value is a human label; unresolved values remain UNKNOWN (not imputed).
final_rows=[]
for r in audit:
    flagged=r['review_id'] in review_reasons
    final_rows.append({
      'review_id':r['review_id'],'cohort_key':r['cohort_key'],'trial_id':r['trial_id'],
      'problem_id':r['problem_id'],'dataset':r['dataset'],'action':r['action'],'role':r['role'],
      'reported_task_success':r['reported_task_success'],
      'human_adjudicated_task_success':resolved.get(r['review_id'],'') if flagged else '',
      'flagged_for_human_review':int(flagged),
      'state':('not_flagged_auto_diagnostic_only' if not flagged else next(s['state'] for s in status if s['review_id']==r['review_id']))
    })
write_csv('adjudicated_vs_reported_all_710.csv',final_rows)
# Assert the logical partial bounds account for every unresolved review case exactly once.
for bound in bounds:
    subset=[x for x in audit if x['cohort']==bound['cohort'] and x['action']==bound['action']
      and (x['cohort_key']!='dream13' or x['role']=='primary_unseen_problem')]
    missing=sum(x['review_id'] in review_reasons and x['review_id'] not in resolved for x in subset)
    assert missing==bound['unresolved_trials']
    assert 0<=bound['minimum_possible_task_success_rate']<=bound['maximum_possible_task_success_rate']<=1
assert sum(x['reported_task_success'] for x in audit)==74
assert len({(x['cohort_key'],x['trial_id']) for x in audit})==710
assert len(review_sheet)==159
print('Corrected unresolved-case identification bounds and composite/fingerprint IDs: PASS')


### 7. Export a reproducible ZIP with input checksums, reviewer-status gate, tables, and reviewer templates

In [ ]:
def export_archive(filename,include_names=None):
    target=OUTPUT_ROOT/filename
    files=sorted(OUT.iterdir()) if include_names is None else [OUT/s for s in include_names]
    files=[p for p in files if p.is_file() and p.name!='SHA256_MANIFEST.json']
    manifest={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in files}
    write_json('SHA256_MANIFEST.json',{'evidence_bundle_sha256':BUNDLE_SHA256,'file_sha256':manifest})
    with zipfile.ZipFile(target,'w',zipfile.ZIP_DEFLATED,compresslevel=8) as z:
        for p in files+[OUT/'SHA256_MANIFEST.json']:z.write(p,arcname=p.name)
    with zipfile.ZipFile(target) as z:
        m=json.loads(z.read('SHA256_MANIFEST.json'))
        for name,digest in m['file_sha256'].items():assert hashlib.sha256(z.read(name)).hexdigest()==digest
    print('Verified results archive:',target,'(',target.stat().st_size,'bytes )')
    if colab_files is not None:
        try:colab_files.download(str(target))
        except Exception as e:print('Automatic download unavailable:',e)
    return target

RESULT_ZIP=export_archive('RepairScopeX_NB25_Final_Arbitrated_20261019_results.zip')

**Interpretation:** Zero completed reviewer files means zero human-verified adjudications, even if automated diagnostics flag an expression as correct. Finish both independent reviews, import files, resolve disputes, and rerun Sections 5–7 to update the results. Do not treat Notebook 21's pending MDLM repair cohort as negative evidence.